# Byte-Pair Encoding Utilities

The reversible byte mapping and ranked merges used by GPT-2 tokenization.

## Imports and defaults

Import shared numerical, model, image, and typing tools before defining reusable concepts.

In [ ]:
#| export
"""GPT-2 byte-pair encoding utilities used by min-DALL-E."""

import json
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import TypedDict

import regex
import requests
import torch
from torch import Tensor

ENCODER_URL: str = "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json"
VOCAB_URL: str = "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe"

## Debug structures and byte vocabulary

Represent tokenization traces and map every byte to reversible Unicode.

In [ ]:
#| export
class BPEPart(TypedDict):
    """Intermediate values for one pre-tokenized text segment."""

    token: str
    token_bytes: bytes
    token_translated: str
    token_merged: list[str]
    token_ix: list[int]


class BPEWork(TypedDict):
    """Debug representation returned by ``encode_and_show_work``."""

    bpe_idx: list[int]
    tokens: list[str]
    parts: list[BPEPart]


def bytes_to_unicode() -> dict[int, str]:
    """Map all byte values to reversible, printable Unicode characters.

    Returns:
        A one-to-one mapping for every integer byte value from 0 through 255.
    """
    byte_values: list[int] = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )
    unicode_values: list[int] = byte_values.copy()
    shifted_index = 0
    for byte_value in range(2**8):
        if byte_value not in byte_values:
            byte_values.append(byte_value)
            unicode_values.append(2**8 + shifted_index)
            shifted_index += 1
    return dict(zip(byte_values, map(chr, unicode_values), strict=True))


def get_pairs(word: Sequence[str]) -> set[tuple[str, str]]:
    """Return all adjacent symbol pairs in a token.

    Args:
        word: Token represented as an ordered symbol sequence.

    Returns:
        Unique adjacent pairs. An empty or one-symbol input has no pairs.
    """
    return set(zip(word, word[1:], strict=False))

## Ranked byte-pair encoder

Split text, merge symbols by rank, and decode IDs losslessly.

In [ ]:
#| export
class Encoder:
    """GPT-2-compatible byte-pair encoder and decoder.

    Args:
        encoder: Mapping from merged BPE symbols to token IDs.
        bpe_merges: Ranked symbol pairs, from highest to lowest priority.
    """

    def __init__(
        self,
        encoder: Mapping[str, int],
        bpe_merges: Sequence[tuple[str, str]],
    ) -> None:
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder: dict[str, int] = {value: key for key, value in self.byte_encoder.items()}
        self.encoder = dict(encoder)
        self.decoder: dict[int, str] = {value: key for key, value in self.encoder.items()}
        self.bpe_ranks: dict[tuple[str, str], int] = dict(
            zip(bpe_merges, range(len(bpe_merges)), strict=True)
        )
        self.pat = regex.compile(
            r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+|"
            r" ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"
        )
        self.cache: dict[str, str] = {}

    def bpe(self, token: str) -> str:
        """Apply ranked byte-pair merges to one byte-encoded token."""
        if token in self.cache:
            return self.cache[token]

        word: tuple[str, ...] = tuple(token)
        pairs: set[tuple[str, str]] = get_pairs(word)
        if not pairs:
            return token

        while pairs:
            bigram: tuple[str, str] = min(
                pairs,
                key=lambda pair: self.bpe_ranks.get(pair, float("inf")),
            )
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            merged_word: list[str] = []
            index = 0
            while index < len(word):
                try:
                    next_first = word.index(first, index)
                except ValueError:
                    merged_word.extend(word[index:])
                    break
                merged_word.extend(word[index:next_first])
                index = next_first
                if index < len(word) - 1 and word[index + 1] == second:
                    merged_word.append(first + second)
                    index += 2
                else:
                    merged_word.append(word[index])
                    index += 1
            word = tuple(merged_word)
            if len(word) == 1:
                break
            pairs = get_pairs(word)

        merged_token: str = " ".join(word)
        self.cache[token] = merged_token
        return merged_token

    def encode(self, text: str) -> list[int]:
        """Encode arbitrary UTF-8 text as GPT-2 BPE token IDs."""
        token_ids: list[int] = []
        for token in self.pat.findall(text):
            translated: str = "".join(self.byte_encoder[byte] for byte in token.encode("utf-8"))
            merged_symbols: list[str] = self.bpe(translated).split(" ")
            token_ids.extend(self.encoder[symbol] for symbol in merged_symbols)
        return token_ids

    def encode_and_show_work(self, text: str) -> BPEWork:
        """Encode text and return each intermediate tokenization step."""
        token_ids: list[int] = []
        parts: list[BPEPart] = []
        tokens: list[str] = self.pat.findall(text)
        for token in tokens:
            token_bytes: bytes = token.encode("utf-8")
            translated: str = "".join(self.byte_encoder[byte] for byte in token_bytes)
            merged_symbols: list[str] = self.bpe(translated).split(" ")
            part_ids: list[int] = [self.encoder[symbol] for symbol in merged_symbols]
            token_ids.extend(part_ids)
            parts.append(
                {
                    "token": token,
                    "token_bytes": token_bytes,
                    "token_translated": translated,
                    "token_merged": merged_symbols,
                    "token_ix": part_ids,
                }
            )
        return {"bpe_idx": token_ids, "tokens": tokens, "parts": parts}

    def decode(self, bpe_idx: Sequence[int]) -> str:
        """Decode GPT-2 BPE token IDs back to UTF-8 text."""
        merged_tokens: list[str] = [self.decoder[token_id] for token_id in bpe_idx]
        flattened: str = "".join(merged_tokens)
        token_bytes = bytearray(self.byte_decoder[character] for character in flattened)
        return token_bytes.decode("utf-8", errors="replace")

## Vocabulary asset loading

Download the official vocabulary only when explicitly requested.

In [ ]:
#| export
def get_file(
    local_file: str | Path,
    remote_file: str,
    timeout: float = 30.0,
) -> None:
    """Download a file only when no local copy exists.

    Args:
        local_file: Destination path.
        remote_file: Source URL.
        timeout: HTTP request timeout in seconds.

    Raises:
        requests.HTTPError: If the server returns an unsuccessful status.
    """
    destination = Path(local_file)
    if destination.is_file():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading {remote_file} to {destination}")
    response = requests.get(remote_file, timeout=timeout)
    response.raise_for_status()
    destination.write_bytes(response.content)


def get_encoder(cache_dir: str | Path | None = None) -> Encoder:
    """Load the official GPT-2 vocabulary and return a BPE encoder.

    Args:
        cache_dir: Optional cache directory. Defaults to ``~/.cache/mingpt``.

    Returns:
        Configured GPT-2-compatible encoder.
    """
    cache = (
        Path(cache_dir).expanduser() if cache_dir is not None else Path.home() / ".cache" / "mingpt"
    )
    cache.mkdir(parents=True, exist_ok=True)

    encoder_file = cache / "encoder.json"
    get_file(encoder_file, ENCODER_URL)
    encoder_data: dict[str, int] = json.loads(encoder_file.read_text(encoding="utf-8"))
    if len(encoder_data) != 50_257:
        raise ValueError("GPT-2 encoder vocabulary must contain 50,257 entries")

    vocab_file = cache / "vocab.bpe"
    get_file(vocab_file, VOCAB_URL)
    bpe_data: str = vocab_file.read_text(encoding="utf-8")
    merge_lines: list[str] = bpe_data.split("\n")[1:-1]
    bpe_merges: list[tuple[str, str]] = [
        (parts[0], parts[1]) for merge_line in merge_lines if len(parts := merge_line.split()) == 2
    ]
    if len(bpe_merges) != 50_000:
        raise ValueError("GPT-2 BPE vocabulary must contain 50,000 merges")
    return Encoder(encoder_data, bpe_merges)

## PyTorch tokenizer adapter

Expose encoded IDs as a batch-first integer tensor.

In [ ]:
#| export
class BPETokenizer:
    """PyTorch-aware wrapper around the GPT-2 BPE encoder."""

    def __init__(self, encoder: Encoder | None = None) -> None:
        """Initialize with a supplied or downloaded encoder."""
        self.encoder = encoder or get_encoder()

    def __call__(self, text: str, return_tensors: str = "pt") -> Tensor:
        """Encode one string as a batch-first integer tensor.

        Raises:
            ValueError: If a tensor format other than ``"pt"`` is requested.
        """
        if return_tensors != "pt":
            raise ValueError("BPETokenizer only supports return_tensors='pt'")
        return torch.tensor([self.encoder.encode(text)], dtype=torch.long)

    def decode(self, idx: Tensor) -> str:
        """Decode a one-dimensional token-ID tensor.

        Raises:
            ValueError: If ``idx`` is not one-dimensional.
        """
        if idx.ndim != 1:
            raise ValueError("decode expects a one-dimensional token tensor")
        return self.encoder.decode(idx.tolist())

## Summary

The tokenizer preserves GPT-2 behavior and performs network access only through an explicit loader.